# 322. Coin Change
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/coin-change/


## Concepts

**Core concept(s):** Recursion, Memoization (top-down DP), and Dynamic Programming / Tabulation (bottom-up DP).

**Why they apply here:** Coin Change asks for the *fewest* coins that sum to a target amount, where each coin denomination can be reused an unlimited number of times (an "unbounded knapsack" style problem). The key structural clue is **optimal substructure**: the best way to make `amount` using coin `c` is `1 + (best way to make amount - c)`. Because the same sub-amount (e.g. "make 7") is reached via many different coin orderings (e.g. 1+1+5 or 5+1+1), naive recursion **recomputes identical subproblems exponentially many times** — this is the signal for memoization / DP.

**Key intuition / mental model:** Think of `amount` as a ladder with `amount` rungs. From rung `a` you can jump down to rung `a - c` for any coin `c`. We want the shortest path from rung `amount` down to rung `0`. Because many paths land on the same rung, we should remember ("memoize") the shortest path already found from each rung instead of re-exploring it.

**Prerequisite knowledge:**
- Comfortable with recursive function calls and call stacks.
- Basic array/list indexing.

---

**What is Recursion?**
- **What it is:** A function that solves a problem by calling itself on smaller version(s) of the same problem, until it reaches a **base case** it can answer directly.
- **How it works internally / mental picture:** Each call pushes a new frame onto the **call stack** holding that call's local variables; when a call hits its base case it returns a value, and each waiting frame combines that value with its own work before it, too, returns and pops off the stack.
- **Key operations + complexity:** A call tree of depth `d` with branching factor `b` does up to `O(b^d)` work if nothing is cached — this is exactly what happens in the brute-force approach below.
- **In Python:** an ordinary function that calls itself, e.g. `def f(n): return f(n-1) + ...`. Python's default recursion limit (~1000) matters for deep recursion on large `amount`.

**What is Memoization (Top-Down DP)?**
- **What it is:** An optimization for recursion where we **cache** the result of each unique sub-problem the first time it's computed, then return the cached value instantly on every later request for that same sub-problem.
- **How it works internally / mental picture:** A dictionary (or array) maps "sub-problem identity" (here, a remaining amount) → "already-computed answer". Before doing real work, the function checks the cache; only a cache-miss does the recursive work, then stores the result before returning.
- **Key operations + complexity:** Cache lookup/insert is **O(1) average** (hash map) or O(1) (array indexed by amount). This collapses the exponential brute-force tree down to **one recursive call per distinct sub-problem**.
- **In Python:** a `dict` keyed by remaining amount, or a pre-sized `list` (since amounts are bounded integers `0..amount`), or the `functools.lru_cache` decorator.

**What is Dynamic Programming (Bottom-Up / Tabulation)?**
- **What it is:** Instead of recursing top-down from the original problem, DP tabulation builds the answer **bottom-up**: solve the smallest sub-problems first, store them in a table, and use already-solved smaller entries to fill in larger ones, iteratively, until the table holds the answer to the full problem.
- **How it works internally / mental picture:** A 1-D array `dp` where `dp[a]` = fewest coins to make amount `a`. We initialize `dp[0] = 0` (zero coins needed for zero amount) and every other entry to "infinity" (unreachable so far). We then walk `a` from `1` up to `amount`, and for each coin `c <= a`, we try `dp[a] = min(dp[a], dp[a - c] + 1)`.
- **Key operations + complexity:** Filling the table is `O(amount * number_of_coins)` time (two nested loops) and `O(amount)` space for the table. No recursion/call-stack overhead, and no risk of Python's recursion-depth limit.
- **In Python:** a `list` of length `amount + 1`, initialized with a large sentinel value (e.g. `float('inf')` or `amount + 1`).

Only one primer is written per concept; later approaches that reuse "memoization" or "DP" refer back to these definitions rather than repeating them.


## Problem Statement

You are given an integer array `coins` representing coins of different denominations, and an integer `amount` representing a total amount of money.

Return the **fewest number of coins** needed to make up that amount. If that amount of money **cannot** be made up by any combination of the coins, return `-1`. You may assume there are **infinitely many coins** of each denomination.

**Example 1:**
```
Input: coins = [1, 2, 5], amount = 11
Output: 3        # 11 = 5 + 5 + 1
```

**Example 2:**
```
Input: coins = [2], amount = 3
Output: -1       # 3 cannot be made using only 2s
```

**Example 3:**
```
Input: coins = [1], amount = 0
Output: 0        # zero coins needed to make amount 0
```

**Constraints:**
- `1 <= coins.length <= 12`
- `1 <= coins[i] <= 2^31 - 1`
- `0 <= amount <= 10^4`


### Approach 1 — Brute Force (Plain Recursion)

**Idea:** For a remaining `amount`, try every coin `c` that is `<= amount`. Recursively solve `amount - c` and take `1 + (best of those results)`. The base case is `amount == 0` (0 coins needed); if `amount < 0` the branch is invalid (return infinity/sentinel). We take the `min` over all coin choices. There is **no caching** — identical sub-amounts reached via different coin orders are recomputed from scratch every time.

**Time complexity:** `O(n^amount)` in the worst case, where `n = len(coins)` — at each of up to `amount` levels of recursion we branch into up to `n` children, and (with coin=1 always valid) the recursion depth can reach `amount`, giving an exponential call tree.

**Space complexity:** `O(amount)` for the recursion call stack depth (no memo table is stored).


In [ ]:
from typing import List

def coin_change_brute(coins: List[int], amount: int) -> int:
    """Brute-force recursion: try every coin at every remaining amount, no caching."""

    def solve(remaining: int) -> float:
        if remaining == 0:
            return 0                      # base case: no coins needed
        if remaining < 0:
            return float('inf')           # invalid path: overshot the amount
        best = float('inf')
        for c in coins:                   # branch on every coin choice
            best = min(best, 1 + solve(remaining - c))  # recompute sub-amounts repeatedly (the flaw)
        return best

    result = solve(amount)
    return result if result != float('inf') else -1


### Approach 2 — Better (Memoized Top-Down DP)

**Idea:** Identical to the brute-force recursion, but before doing any work for a given `remaining` amount, check a cache (`memo`). If we've already solved this exact `remaining` before, return the cached answer instantly instead of re-branching into the whole subtree again. Store each newly computed answer in the cache before returning it. This is the same recursion tree, but every distinct `remaining` value is only ever computed **once**.

**Time complexity:** `O(amount * n)` where `n = len(coins)` — there are only `amount + 1` distinct sub-problems (`remaining` ranges over `0..amount`), and each does `O(n)` work trying every coin.

**Space complexity:** `O(amount)` for the memo table plus `O(amount)` for the recursion call stack (both bounded by the number of distinct sub-problems / max recursion depth).


In [ ]:
from typing import List, Dict

def coin_change_memo(coins: List[int], amount: int) -> int:
    """Top-down DP: same recursion as brute force, but cache each remaining-amount result."""
    memo: Dict[int, float] = {}

    def solve(remaining: int) -> float:
        if remaining == 0:
            return 0
        if remaining < 0:
            return float('inf')
        if remaining in memo:             # cache hit: this sub-problem was already solved
            return memo[remaining]
        best = float('inf')
        for c in coins:
            best = min(best, 1 + solve(remaining - c))
        memo[remaining] = best            # cache the result before returning
        return best

    result = solve(amount)
    return result if result != float('inf') else -1


### Approach 3 — Optimal (Bottom-Up Tabulation)

**Idea:** Build a table `dp` of size `amount + 1`, where `dp[a]` holds the fewest coins needed to make amount `a`. Start with `dp[0] = 0` and every other slot set to a large sentinel (unreachable so far). Walk `a` from `1` up to `amount`; for each coin `c <= a`, relax `dp[a] = min(dp[a], dp[a - c] + 1)` using the already-finalized smaller entry `dp[a - c]`. By the time we reach `dp[amount]`, all smaller answers it depends on are already computed. This removes recursion entirely — no call stack, no Python recursion-limit risk — while keeping the same asymptotic work as the memoized version.

**Time complexity:** `O(amount * n)` where `n = len(coins)` — one pass over `amount` values, each doing `O(n)` work trying every coin.

**Space complexity:** `O(amount)` for the `dp` array (no call stack, so noticeably smaller constant-factor memory and better real-world performance than Approach 2 on large `amount`).


In [ ]:
from typing import List

def coin_change_optimal(coins: List[int], amount: int) -> int:
    """Bottom-up DP (tabulation): fill dp[0..amount] iteratively from smallest to largest."""
    INF = amount + 1                      # sentinel: larger than any real answer (max coins = amount, using all 1s)
    dp = [INF] * (amount + 1)
    dp[0] = 0                             # base case: 0 coins to make amount 0

    for a in range(1, amount + 1):        # build up smaller amounts first
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)  # use already-finalized dp[a - c]

    return dp[amount] if dp[amount] != INF else -1


## Test / Verification

Run all three approaches against the examples above plus a couple of edge cases (unreachable amount, `amount = 0`, a single-denomination coin set) and confirm they agree.


In [ ]:
test_cases = [
    ([1, 2, 5], 11, 3),
    ([2], 3, -1),
    ([1], 0, 0),
    ([1], 1, 1),
    ([1, 2, 5], 100, 20),
    ([186, 419, 83, 408], 6249, 20),
]

approaches = {
    "brute":   coin_change_brute,
    "memo":    coin_change_memo,
    "optimal": coin_change_optimal,
}

for coins, amount, expected in test_cases:
    for name, fn in approaches.items():
        # skip the exponential brute force on the large stress case to keep this cell fast
        if name == "brute" and amount > 20:
            continue
        got = fn(coins, amount)
        status = "OK" if got == expected else "FAIL"
        print(f"[{status}] {name:8s} coins={coins} amount={amount} -> {got} (expected {expected})")
        assert got == expected, f"{name} failed for coins={coins}, amount={amount}: got {got}, expected {expected}"

print("\nAll assertions passed.")


## Empirical Complexity Benchmark

Big-O can't be read directly off the code, but it can be **measured**: time each approach on inputs of growing size `n` and see how runtime scales when `n` doubles.

- **Approach 1 (Brute force)** is exponential in `amount` — runtime should grow **far faster than 2x** (often 4x-8x+) each time `amount` doubles, so we only test it on small amounts.
- **Approach 2 (Memoized)** and **Approach 3 (Bottom-up)** are both `O(amount * n_coins)` — with `n_coins` fixed, runtime should grow **~2x** each time `amount` doubles (linear in `amount`).

| Growth pattern when n doubles | Implied complexity |
|---|---|
| ~1x | O(1) / O(log n) |
| ~2x | O(n) / O(n log n) |
| ~4x | O(n^2) |
| ~8x+ or worse | O(n^3) or exponential |

**Note on the test setup:** the DP approaches (memoized and bottom-up) use `coins = [1]` as their worst case, since it forces the maximum number of distinct sub-problems / table entries for a given `amount` (no larger denomination can shortcut to the target). Because that also forces the memoized approach's recursion depth to equal `amount`, the benchmark cell raises Python's recursion limit before timing it — a Python implementation detail, not part of the algorithm's actual complexity.

The **uncached brute force**, however, needs a *different* worst case: with only `coins = [1]` there is just one branch per call (no real branching), so no exponential behavior would show up at all! To see genuine exponential blow-up we use `coins = [1, 3, 4, 5]` — a set with no dominant coin that can trivially short-circuit the search — and deliberately much smaller `amount` sizes, since the runtime otherwise explodes far too fast to finish.


In [ ]:
import os
import sys

# Bootstrap: locate bench_utils.py by walking up from the current working directory,
# so this cell works regardless of which folder the notebook kernel starts in.
# NOTE (test-run deviation): in the real skill output location this walk finds the
# single shared `notebooks/bench_utils.py`. For this test run, bench_utils.py was
# instead placed alongside this notebook in the same outputs/ folder, so the walk
# finds it one directory (or zero directories) up rather than at the notebooks root.
def _find_bench_utils(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(6):                      # walk up a bounded number of levels
        candidate = os.path.join(cur, "bench_utils.py")
        if os.path.isfile(candidate):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError("bench_utils.py not found by walking up from " + start)

_bench_dir = _find_bench_utils(os.getcwd())
if _bench_dir not in sys.path:
    sys.path.insert(0, _bench_dir)

from bench_utils import benchmark

# coins = [1] forces the deepest possible recursion for coin_change_memo (depth == amount),
# since no larger denomination can shortcut toward the target. Raise Python's recursion
# limit so the memoized (top-down) approach can complete on the larger sizes below —
# this is purely a Python implementation limit, not part of the algorithm's complexity.
sys.setrecursionlimit(10000)

def make_worst_case_dp(n: int):
    # coins = [1] forces the maximum number of table entries / recursive sub-problems
    # for a given amount (no larger denomination can shortcut toward the target).
    return ([1], n)

def make_worst_case_brute(n: int):
    # For the *uncached* brute force, coins = [1] only gives a single branch per call
    # (no real branching, so no exponential blow-up shows up). To see the true
    # exponential behavior we need multiple denominations with no dominant/early-exit
    # coin -- [1, 3, 4, 5] has no coin that trivially short-circuits the search, so the
    # recursion tree branches genuinely at every level.
    return ([1, 3, 4, 5], n)

# Bottom-up and memoized DP are both O(amount * n_coins); test them on larger amounts.
dp_solutions = {
    "memo (top-down)":    coin_change_memo,
    "bottom-up (optimal)": coin_change_optimal,
}
dp_sizes = [500, 1000, 2000, 4000]
benchmark(dp_solutions, make_worst_case_dp, dp_sizes, plot=True)

# Brute force is exponential; use much smaller sizes (with genuine branching, see above)
# so it finishes in a reasonable time while still clearly showing non-polynomial growth.
brute_solutions = {
    "brute force": coin_change_brute,
}
brute_sizes = [10, 15, 20, 25]
benchmark(brute_solutions, make_worst_case_brute, brute_sizes, plot=False)


## 🧩 Patterns Learned

- **Unbounded Knapsack / "fewest steps to a target" DP.** Whenever a problem lets you reuse items an unlimited number of times and asks for a min/max count or min/max value to hit an exact target, model it as: `dp[target] = best over each item i of (1 + dp[target - item_i])`, with `dp[0]` as the base case.
- **When to reach for this pattern:** the problem statement mentions "fewest/minimum number of X to reach Y," items can be reused without limit, and there's clear optimal substructure (the best answer for a bigger target is built from the best answer for a smaller one).
- **Recursion → Memoization → Tabulation is a standard three-step refinement.** Start with the natural recursive recurrence to get correctness, add a memo dict/array once you notice repeated sub-problems, then convert to a bottom-up table once the sub-problem order is clear (usually smallest-to-largest) to drop recursion overhead and stack-depth risk.
- **Related problems:** Coin Change II (LC 518, count combinations instead of min count), Perfect Squares (LC 279, same shape with `coin_i = i^2`), Minimum Cost For Tickets (LC 983), Word Break (LC 139, boolean instead of min-count DP).
- **Common pitfalls:**
  - Forgetting the `amount == 0 -> 0` base case, or not guarding `remaining < 0` before recursing.
  - Off-by-one on the sentinel value in tabulation (`amount + 1` is a safe "infinity" since the true answer can never exceed making the whole amount out of 1-coins).
  - Testing the brute-force approach on inputs large enough that its exponential blow-up makes the notebook hang.
  - Confusing this "unbounded" pattern (each coin reusable) with the 0/1 knapsack pattern (each item usable at most once) — the loop order and recurrence differ.
